# Telugu FastSpeech 2: data, alignment, training

Run the setup cell first in each session. Cells can be rerun safely; complete downloads on Drive are reused.

In [ ]:
# Run this cell at the start of every Colab session. Runtime type must be GPU.
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/FastSpeech2-IIITH'
REPO = '/content/FastSpeech2-IIITH'
REPO_URL = ''  # optional: GitHub URL once the repository is public
os.makedirs(DRIVE, exist_ok=True)
if not os.path.isdir(REPO):
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
    else:
        subprocess.run(['unzip', '-q', f'{DRIVE}/FastSpeech2-IIITH-code.zip', '-d', REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, REPO)
from src.data.storage import configure_environment
# HF_HOME is deliberately left alone, so Hugging Face's token file stays off Drive.
configure_environment(DRIVE)
os.environ.update(FS2_REPO_ROOT=REPO, PYTHONUNBUFFERED='1')
!nvidia-smi
import torch
print('PyTorch', torch.__version__, 'CUDA build', torch.version.cuda, 'GPU available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU'


In [ ]:
# IndicTTS Telugu is public, so a login is optional here. It only raises rate limits.
from src.data.hf_auth import authenticate
try:
    authenticate()
except RuntimeError as error:
    print('Continuing without a login:', error)


## 1. Verify access and inspect

Runs a real load, measures every clip and checks any existing export. It saves `experiments/telugu/dataset_inspection.json` on Drive. Never commit that file, because it holds transcript samples.

In [ ]:
!python Telugu/preprocess.py --stop inspect

## 2. Restore prepared data and review alignment

Use this if a `prepared.tar` is on Drive under `processed/telugu/`. **Stop and review before going on:** read the coverage, rejection reasons, enough-audio line and printed report. 100% coverage is not required, but a high or systematic rejection rate needs investigating. An archive has TextGrids but no WAV files, so this cell has not checked timing against audio. Timing evidence comes only from `scripts/verify_alignment_timing.py` and spectrogram inspection where the audio exists.

In [ ]:
!python scripts/drive_sync.py restore telugu
!python scripts/check_alignments.py telugu; echo checker exit $?
!cat "$FS2_DATA_ROOT/processed/telugu/report.json"

## 3. Select, align and extract features

Alignment uses the CPU. Review the inspection summary first.

In [ ]:
!python Telugu/preprocess.py --start select --stop align
!python scripts/check_alignments.py telugu; echo checker exit $?
!python scripts/verify_alignment_timing.py telugu --clips 200

**Stop and review the alignment results above.** Run the next cell only if the rejection rate and representative alignments look acceptable.

In [ ]:
!python Telugu/preprocess.py --start features
!python scripts/drive_sync.py save telugu

## 4. Train

After a disconnect, rerun setup, restore, then this cell. It resumes from the newest checkpoint on Drive.

In [ ]:
!python Telugu/train.py 2>&1 | tee -a "$FS2_EXP_ROOT/telugu_train_console.log"